# Construye 2 · Notebook de equipo | Especificación ejecutada
**Módulo 2 · Distribuidora Cuauhtémoc · trabajo por equipo con datos comunes**

El propósito de esta práctica es pasar del contrato de M1 a decisiones que puedan comprobarse: qué caso merece atención, qué debe verificarse y cuándo corresponde detenerse. Su documento de Construye 2 declara la especificación; este notebook permite **comprobar parte de esa especificación** en un simulador sin LLM. El evaluador compara el documento con los registros de ejecución (trazas). Distingan lo implementado de lo propuesto y reporten el desenlace que obtengan: éxito, escalamiento, imposibilidad o paro por límite.

| Sección de su documento | Dónde se ejecuta aquí |
|---|---|
| 1.1 Objetivo operacionalizado | `DIAS_MINIMOS_QUIEBRE`, `VENTANA_QUIEBRES_DIAS`, `UMBRAL_COSTO_MINIMO` |
| 1.2 Entradas y vigencia | `VENTANAS_VIGENCIA`: conteos, no filtrado del detector (Ejecución 1) |
| 1.3 Restricciones | `ACCIONES_PERMITIDAS` (caso adversarial) |
| 1.4 Criterios de éxito | `verificar_criterios()` antes de escribir (Ejecución 1) |
| 1.5 Condiciones de paro | Resultado real de Ej. 1; escalamiento e imposibilidad (Ej. 2); límite ilustrado en la ruptura opcional |
| 1.6 Incertidumbre | Conteos de capturas caducas registrados (Ejecución 1) |

**Entrega:** incorporen en su PDF las decisiones y las trazas de esta ejecución, legibles y con su interpretación. El notebook es su archivo de trabajo; una liga editable no sustituye la evidencia del PDF.

**Requisitos:** el ZIP `Construye_2.zip` incluye este notebook, `equipo_lib.py` y los siete CSV en `datos/`. Todos los equipos usan los mismos datos; sus parámetros y decisiones provienen de su propio contrato de M1, que no necesitan rehacer.


**Alcance del simulador:** la bandeja contiene borradores en memoria, no pedidos enviados. El tope impide nuevas escrituras al agotarse, pero no interrumpe toda la computación de Python. Las comprobaciones automáticas no sustituyen la revisión del documento ni certifican un sistema real.
Si sus umbrales dejan cero candidatos, la corrida declara imposibilidad sin escribir ni fabricar un caso. Las pruebas que necesitan un candidato se reportan como no evaluadas: no se muestran como aprobadas. Un solo candidato basta para el control de escalamiento, y cualquier conflicto agregado para ese control se identifica expresamente como simulado.

En esta etapa, las ventanas registran conteos de vigencia en la traza, pero no filtran todavía los datos usados por el detector. El filtro operativo se integra en el Capstone: allí la frescura del último inventario se separa del historial de quiebres. Distingan en el PDF la política diseñada de lo que este simulador efectivamente ejecuta.

**Cómo abrirlo en Colab**
1. Descarguen `Construye_2.zip` de esta actividad y descomprímanlo en su computadora.
2. En Colab, elijan Archivo → Subir notebook y abran `construye2_agente_equipo.ipynb`.
3. En el panel Archivos de Colab, suban el ZIP completo `Construye_2.zip` (sin cambiarle el nombre). No es necesario subir cada CSV.
4. Ejecuten la primera celda: extrae la biblioteca y los datos del paquete. Después ejecuten las celdas en orden. Si reinician el entorno de Colab, vuelvan a subir el ZIP.

**Ejecución local:** descompriman el paquete completo en una carpeta exclusiva de esta actividad y abran el notebook allí, conservando `datos/`. Se requiere Python con pandas. No mezclen bibliotecas de paquetes anteriores.

In [1]:
# Preparación del paquete común (no requiere cambios del equipo)
from pathlib import Path
import zipfile

RUTA_DATOS = "./datos"
ESPERADOS = ["tiendas", "productos", "ventas", "inventarios", "precios_competencia", "promociones", "visitas"]
necesarios = ["equipo_lib.py"] + [f"datos/{t}.csv" for t in ESPERADOS]
paquete = Path("Construye_2.zip")
if paquete.is_file():
    # Solo se extraen los ocho archivos conocidos, en el directorio de esta sesión.
    # El notebook que están editando no se sobrescribe.
    with zipfile.ZipFile(paquete) as z:
        faltan_zip = [n for n in necesarios if n not in z.namelist()]
        if faltan_zip:
            raise ValueError("El ZIP está incompleto: " + ", ".join(faltan_zip))
        for nombre in necesarios:
            destino = Path(nombre)
            destino.parent.mkdir(parents=True, exist_ok=True)
            destino.write_bytes(z.read(nombre))
faltan = [n for n in necesarios if not Path(n).is_file()]
if faltan:
    raise FileNotFoundError("Faltan archivos: " + ", ".join(faltan) +
        ". En Colab, suban Construye_2.zip al panel Archivos y vuelvan a ejecutar esta celda. "
        "En su computadora, conserven la estructura del ZIP en una carpeta propia.")

import importlib
import equipo_lib
importlib.reload(equipo_lib)
from equipo_lib import (cargar_variante, detectar_quiebres, promociones_activas,
                        capturas_vigentes, recortes_de_vigencia, Traza, Bandeja)
d = cargar_variante(RUTA_DATOS)
print("Conjunto común cargado:", {t: len(d[t]) for t in ESPERADOS})


Conjunto común cargado: {'tiendas': 200, 'productos': 60, 'ventas': 649563, 'inventarios': 475800, 'precios_competencia': 3596, 'promociones': 141, 'visitas': 1950}


## Las decisiones de su documento, ejecutables

En la siguiente celda definan los parámetros de su especificación. Más adelante también completarán la acción prohibida del caso adversarial y la declaración de uso de IA. La celda de ruptura es opcional. No necesitan modificar la biblioteca ni los datos.

Cada valor debe coincidir con su documento. Las comprobaciones ayudan a detectar inconsistencias; no asignan la calificación. Las ventanas cuentan registros de antigüedad entre 0 y N días, ambos incluidos, respecto al corte de cada fuente; todavía no filtran el detector.

### Cómo leer los parámetros

`DIAS_MINIMOS_QUIEBRE` cuenta fechas con inventario cero dentro de la ventana; **no exige días consecutivos**. `VENTANA_QUIEBRES_DIAS` define cuántos días de inventario se observan hasta el corte del dataset, no hasta la fecha en que ustedes ejecutan el notebook.

El costo estimado se calcula así:

**días en cero × (rotación semanal esperada ÷ 7) × precio de lista**.

La rotación viene de la tabla de productos; se aproxima una demanda diaria constante y se redondea el resultado a la unidad monetaria. Por ejemplo, 3 días × (14 unidades/semana ÷ 7) × 100 pesos/unidad = 600 pesos. Es una estimación para priorizar casos, **no una pérdida contable comprobada ni un cálculo de margen**. El umbral debe justificarse con esta escala y con su contrato.

Conserven las tres claves de `VENTANAS_VIGENCIA`; cambien sus valores enteros en días. Las vigencias cuentan edades de 0 a N, ambos incluidos; la ventana del detector selecciona los últimos N días. Son controles distintos y no deben confundirse.

### Las variables que van a encontrar

| Variable | Qué representa |
|---|---|
| `d` | Las siete tablas cargadas. Se accede a cada una por su nombre. |
| `q` | Casos tienda-producto que cumplen los umbrales, ordenados por costo estimado. |
| `top` | Primer caso de esa lista; si la lista está vacía, no se inventa uno. |
| `tr1`, `tr2`, `tr3` | Registros de cada prueba: qué se hizo, con qué evidencia y cómo terminó. |
| `b1`, `b2`, `b3` | Bandejas independientes en memoria; no son pedidos ni mensajes enviados. |

No necesitan modificar la biblioteca. Definan los parámetros, redacten la acción prohibida y completen la declaración de IA; el resto sirve para leer, ejecutar e interpretar.


In [2]:
import math
from datetime import date
# --- DECISIONES DEL EQUIPO (deben coincidir con su documento) ---
DIAS_MINIMOS_QUIEBRE  = 2        # número de fechas con inventario cero; no necesariamente consecutivas
VENTANA_QUIEBRES_DIAS = 7       # últimos N días de inventario respecto al corte de los datos
UMBRAL_COSTO_MINIMO   = 500.0    # estimación monetaria mínima para priorizar un caso

VENTANAS_VIGENCIA = {"ventas": 14, "inventarios": 1, "precios_competencia": 10}
ACCIONES_PERMITIDAS = {"reposicion urgente (aprueba gerente)",
                       "ajuste de pedido sugerido",
                       "alerta a preventista"}
# Tope total del registro, incluido el cierre; entero de 5 a 30.
MAX_PASOS = 12

def validar_politica():
    """Revisa tipos y rangos básicos. No lee ni califica el contrato o el PDF."""
    problemas = []
    if type(DIAS_MINIMOS_QUIEBRE) is not int or type(VENTANA_QUIEBRES_DIAS) is not int:
        problemas.append("DIAS_MINIMOS_QUIEBRE y VENTANA_QUIEBRES_DIAS deben ser enteros")
    elif not (1 <= DIAS_MINIMOS_QUIEBRE <= VENTANA_QUIEBRES_DIAS):
        problemas.append("DIAS_MINIMOS_QUIEBRE debe ser >= 1 y caber en VENTANA_QUIEBRES_DIAS")
    if type(UMBRAL_COSTO_MINIMO) not in (int, float) or not math.isfinite(UMBRAL_COSTO_MINIMO) or UMBRAL_COSTO_MINIMO <= 0:
        problemas.append("UMBRAL_COSTO_MINIMO debe ser un número finito positivo")
    fuentes = {"ventas", "inventarios", "precios_competencia"}
    if not isinstance(VENTANAS_VIGENCIA, dict) or set(VENTANAS_VIGENCIA) != fuentes:
        problemas.append("VENTANAS_VIGENCIA debe conservar ventas, inventarios y precios_competencia")
    elif any(type(dias) is not int or dias < 1 for dias in VENTANAS_VIGENCIA.values()):
        problemas.append("Cada ventana de vigencia debe ser un entero >= 1 día")
    if not isinstance(ACCIONES_PERMITIDAS, (set, list, tuple)) or not ACCIONES_PERMITIDAS or not all(isinstance(a, str) and a.strip() for a in ACCIONES_PERMITIDAS):
        problemas.append("ACCIONES_PERMITIDAS debe ser una colección no vacía de textos de acciones")
    if type(MAX_PASOS) is not int or not 5 <= MAX_PASOS <= 30:
        problemas.append("MAX_PASOS debe ser un entero de 5 a 30, incluido el registro de cierre")
    if problemas:
        raise ValueError("Revisen los parámetros:\n- " + "\n- ".join(problemas))
    print("Parámetros básicos validados; cotejen sus valores con el contrato y el documento.")
    print(f"Quiebre >= {DIAS_MINIMOS_QUIEBRE} días; ventana {VENTANA_QUIEBRES_DIAS} días; costo >= {UMBRAL_COSTO_MINIMO}")
    print(f"Vigencias: {VENTANAS_VIGENCIA}; acciones permitidas: {sorted(ACCIONES_PERMITIDAS)}")
    print(f"Tope total: {MAX_PASOS} registros; el último espacio se reserva para el cierre.")
    return True

POLITICA_OK = validar_politica()

def fecha_iso_valida(valor):
    """Acepta una fecha real YYYY-MM-DD. No demuestra que sea vigente o verdadera."""
    if not isinstance(valor, str):
        return False
    try:
        return date.fromisoformat(valor).isoformat() == valor
    except ValueError:
        return False

def verificar_criterios(caso, tr=None):
    """Comprueba criterios programados antes de escribir; no sustituye todos los del contrato."""
    costo = caso.get("costo")
    checks = {
        "identificadores_presentes": bool(caso.get("tienda_id")) and bool(caso.get("producto_id")),
        "fecha_corte_valida": fecha_iso_valida(caso.get("fecha_corte")),
        "costo_supera_umbral": type(costo) in (int, float) and math.isfinite(costo) and costo >= UMBRAL_COSTO_MINIMO,
        "accion_dentro_del_contrato": caso.get("accion_propuesta") in ACCIONES_PERMITIDAS,
    }
    ok = all(checks.values())
    if tr: tr.paso("verificar", "criterios programados satisfechos antes de escribir" if ok else "BLOQUEADO antes de la bandeja: criterios incumplidos", checks)
    return ok


Parámetros básicos validados; cotejen sus valores con el contrato y el documento.
Quiebre >= 2 días; ventana 7 días; costo >= 500.0
Vigencias: {'ventas': 14, 'inventarios': 1, 'precios_competencia': 10}; acciones permitidas: ['ajuste de pedido sugerido', 'alerta a preventista', 'reposicion urgente (aprueba gerente)']
Tope total: 12 registros; el último espacio se reserva para el cierre.


## Ejecución 1 · Evaluación de un caso completo

Con los valores iniciales se prepara un borrador. Con sus propias decisiones puede haber éxito, escalamiento, bloqueo, imposibilidad o paro por límite. Reporten el estado real; no cambien sus umbrales solo para obtener verde.

Un paso es una entrada del registro; una vuelta es un recorrido por las fases. `MAX_PASOS` incluye el cierre: se reserva el último espacio para documentar cómo termina la ejecución. Si no queda espacio para escribir y registrar su cierre, no se escribe. El aviso final **no rebasa el tope**.

### Relación con los Aprende 3.1–3.5

| Fase o decisión | Qué hace este código | Qué deben interpretar |
|---|---|---|
| Observar | Cuenta vigencias y busca inventarios en cero. | Qué datos usa y qué limitaciones conserva. |
| Razonar y planear | Prioriza por costo y consulta promociones. | Por qué selecciona ese caso y cuándo aparece un conflicto. Son reglas programadas, no razonamiento de un LLM. |
| Verificar antes de actuar | Comprueba identificadores presentes, fecha válida, costo y acción autorizada; la bandeja también valida que tienda y producto existan. | Qué controles previenen una escritura incorrecta y qué criterios de su contrato aún son de diseño. |
| Actuar y cerrar | Escribe solo un borrador permitido y confirma el resultado, o registra por qué se detuvo. | Qué pasó, con qué evidencia y qué decisión queda al humano. |

Antes de ejecutar, anticipen qué efecto tendrán sus parámetros. Después contrasten esa expectativa con la salida y expliquen la diferencia dentro de la interpretación del PDF ya solicitado; no es una entrega adicional. La biblioteca se encarga de la infraestructura, no de justificar las decisiones por ustedes.


In [3]:
tr1 = Traza("C2-caso-completo", tope_pasos=MAX_PASOS)

recortes_de_vigencia(d, VENTANAS_VIGENCIA, tr=tr1)
vig, cad = capturas_vigentes(d, VENTANAS_VIGENCIA["precios_competencia"], tr=tr1)

q = detectar_quiebres(d, DIAS_MINIMOS_QUIEBRE, VENTANA_QUIEBRES_DIAS, tr=tr1)
q = q[q.costo_estimado >= UMBRAL_COSTO_MINIMO].reset_index(drop=True)
top = None
b1 = Bandeja(d)
fecha_corte = str(d["inventarios"].fecha.max().date())
if q.empty:
    r1 = "imposibilidad"
    tr1.paso("paro", "IMPOSIBILIDAD: ningún caso califica; no se inventa una propuesta",
             {"umbral": UMBRAL_COSTO_MINIMO, "casos": 0})
else:
    top = q.iloc[0]
    tr1.paso("razonar", "quiebre más costoso por encima del umbral del equipo",
             {"caso": f"{top.tienda_id}/{top.producto_id}", "costo": float(top.costo_estimado),
              "umbral": UMBRAL_COSTO_MINIMO, "capturas_caducas_declaradas": len(cad)})

    zona = d["tiendas"].set_index("tienda_id").loc[top.tienda_id, "zona"]
    promos = promociones_activas(d, zona)
    conflicto = bool((promos.producto_id == top.producto_id).any())
    tr1.paso("planear", "conflicto con promociones activas de la zona verificado",
             {"zona": zona, "promos_activas": int(len(promos)), "conflicto": conflicto})

    caso = dict(tienda_id=top.tienda_id, producto_id=top.producto_id, costo=float(top.costo_estimado),
                fecha_corte=fecha_corte, accion_propuesta="reposicion urgente (aprueba gerente)",
                conflicto=conflicto, zona=zona,
                evidencia_conflicto={"origen": "datos comunes",
                                     "promociones": promos.loc[promos.producto_id == top.producto_id, ["promo_id", "tipo", "estado"]].to_dict("records")})
    criterios_ok = verificar_criterios(caso, tr1)

    b1 = Bandeja(d)
    r1 = b1.escribir_caso(caso, tr=tr1) if criterios_ok else "bloqueada_por_criterios"
    if r1 == "aceptada":
        tr1.paso("paro", "ÉXITO: caso completo en bandeja con criterios verificados", {})
tr1.mostrar()
print("Estado final:", r1, "| borradores escritos:", len(b1.casos))
print(f"\nPresupuesto usado: {tr1.presupuesto_usado}/{MAX_PASOS}")


----- Traza C2-caso-completo -----
[ 1] OBSERVAR     ventas: ventana 14d; 107814 registros vigentes, 541749 fuera de ventana (declarados) | evidencia: {'corte': '2026-08-30'}
[ 2] OBSERVAR     inventarios: ventana 1d; 8400 registros vigentes, 467400 fuera de ventana (declarados) | evidencia: {'corte': '2026-08-30'}
[ 3] OBSERVAR     precios_competencia: ventana 10d; 478 registros vigentes, 3118 fuera de ventana (declarados) | evidencia: {'corte': '2026-08-28'}
[ 4] OBSERVAR     capturas: 478 vigentes (<= 10d), 3118 caducas identificadas (conteo, sin filtrar el detector) | evidencia: {'corte': '2026-08-28'}
[ 5] OBSERVAR     inventarios ventana 7d; quiebres >= 2d en cero | evidencia: {'combinaciones': 1600, 'corte': '2026-08-30'}
[ 6] RAZONAR      quiebre más costoso por encima del umbral del equipo | evidencia: {'caso': 'T0046/P060', 'costo': 2719.0, 'umbral': 500.0, 'capturas_caducas_declaradas': 3118}
[ 7] PLANEAR      conflicto con promociones activas de la zona verificado | evidenc

## Ejecución 2 · Escalamiento e imposibilidad

Son **dos controles independientes** del mismo diseño, reunidos en un registro para facilitar su lectura; no son dos finales simultáneos de un solo caso.

Primero se busca, entre los primeros 50 candidatos, un producto con una promoción marcada como activa en su zona. La salida muestra el caso, la promoción y la decisión que requiere revisión humana. Se utiliza el estado de la promoción incluido en el dataset; no se consulta un calendario comercial real. Si esos candidatos no contienen un conflicto, el control se marca expresamente como simulado y no se inventa una promoción.

Después se prueba un umbral que ningún candidato alcanza para demostrar imposibilidad. Ese umbral pertenece al control, no sustituye la decisión que ustedes justificaron en el documento. Escalar prepara evidencia; aquí no se envía ningún aviso al gerente.


In [4]:
tr2 = Traza("C2-escala-e-imposibilidad", tope_pasos=MAX_PASOS)
b2 = Bandeja(d)
if q.empty:
    r2 = "sin_candidato"
else:
    tz = d["tiendas"].set_index("tienda_id")["zona"]
    caso_conflicto = None
    for _, fila in q.head(50).iterrows():
        coincidencias = promociones_activas(d, tz.loc[fila.tienda_id])
        coincidencias = coincidencias[coincidencias.producto_id == fila.producto_id]
        if not coincidencias.empty:
            caso_conflicto = fila
            break
    if caso_conflicto is None:
        caso_conflicto = q.iloc[0]
        evidencia_conflicto = {"origen": "control SIMULADO; no detectado entre los candidatos revisados", "promociones": []}
    else:
        evidencia_conflicto = {"origen": "datos comunes", "promociones": coincidencias[["promo_id", "tipo", "estado"]].to_dict("records")}
    caso_para_escalar = dict(tienda_id=caso_conflicto.tienda_id, producto_id=caso_conflicto.producto_id,
        zona=tz.loc[caso_conflicto.tienda_id], costo=float(caso_conflicto.costo_estimado), fecha_corte=fecha_corte,
        accion_propuesta="reposicion urgente (aprueba gerente)", conflicto=True, evidencia_conflicto=evidencia_conflicto)
    r2 = b2.escribir_caso(caso_para_escalar, tr=tr2)

# Control separado: ninguna combinación supera este umbral de prueba.
umbral_imposible = float(q.costo_estimado.max()) * 1000 if not q.empty else UMBRAL_COSTO_MINIMO
q_imposible = q[q.costo_estimado >= umbral_imposible]
if len(q_imposible) == 0:
    tr2.paso("paro", "IMPOSIBILIDAD declarada en un control independiente: ningún caso califica",
             {"umbral_probado": umbral_imposible, "casos": 0})
tr2.mostrar()
print(f"\nConflicto: {r2}; imposibilidad declarada sin rellenar: {len(q_imposible) == 0}")


----- Traza C2-escala-e-imposibilidad -----
[ 1] PARO         ESCALAMIENTO: caso preparado para revisión humana (sin notificación real) | evidencia: {'tienda': 'T0060', 'producto': 'P056', 'zona': 'Bajio', 'costo_estimado': 1828.0, 'fecha_corte': '2026-08-30', 'accion_propuesta': 'reposicion urgente (aprueba gerente)', 'conflicto': {'origen': 'datos comunes', 'promociones': [{'promo_id': 'PR900', 'tipo': 'descuento_pct', 'estado': 'activa'}]}, 'decision_pendiente': 'El gerente debe revisar el conflicto antes de autorizar; no se escribió el borrador.'}
[ 2] PARO         IMPOSIBILIDAD declarada en un control independiente: ningún caso califica | evidencia: {'umbral_probado': 2719000.0, 'casos': 0}

Conflicto: escalada; imposibilidad declarada sin rellenar: True


## Caso adversarial · La acción fuera del contrato

Su documento (§1.3 y decisión «dónde verificar») dice que el agente verifica **antes** de escribir. Aquí lo prueban: **redacten una acción que su contrato prohíbe** y demuestren que el verificador la bloquea antes de tocar la bandeja. Una acción genérica no demuestra nada: piensen qué propondría un agente mal configurado en su dominio.


In [5]:
# --- DECISIÓN: redacten la acción prohibida por SU contrato de M1 ---
ACCION_FUERA_DE_CONTRATO = "enviar pedido directo al proveedor sin aprobacion del gerente"

if ACCION_FUERA_DE_CONTRATO in ACCIONES_PERMITIDAS:
    raise ValueError("Esa acción está en su lista de permitidas: elijan una que su contrato de M1 prohíba.")

tr3 = Traza("C2-adversarial", tope_pasos=MAX_PASOS)
b3 = Bandeja(d)
bloqueado = None
if top is None:
    tr3.paso("paro", "NO EVALUADO: sin candidato no se demuestra el caso adversarial", {})
else:
    caso_adv = dict(tienda_id=top.tienda_id, producto_id=top.producto_id, costo=float(top.costo_estimado),
                    fecha_corte=fecha_corte, accion_propuesta=ACCION_FUERA_DE_CONTRATO, conflicto=False)
    bloqueado = not verificar_criterios(caso_adv, tr3)
    b3 = Bandeja(d)
    if not bloqueado:
        b3.escribir_caso(caso_adv, tr=tr3)
tr3.mostrar()
print(f"\nAcción fuera de contrato bloqueada antes de la bandeja: {bloqueado}; casos escritos: {len(b3.casos)}")


----- Traza C2-adversarial -----
[ 1] VERIFICAR    BLOQUEADO antes de la bandeja: criterios incumplidos | evidencia: {'identificadores_presentes': True, 'fecha_corte_valida': True, 'costo_supera_umbral': True, 'accion_dentro_del_contrato': False}

Acción fuera de contrato bloqueada antes de la bandeja: True; casos escritos: 0


## La celda de ruptura (opcional, no se entrega)

Estos controles muestran mensajes cuando se incumple una condición: un umbral inválido, un presupuesto insuficiente o una fecha ausente o inválida. Lean qué detecta cada control; no confundan una prueba rechazada correctamente con una falla del notebook. Al terminar se restauran los parámetros de su equipo.


In [6]:
# 1) Umbral en cero: el validador explica por qué no
try:
    _u = UMBRAL_COSTO_MINIMO
    UMBRAL_COSTO_MINIMO = 0
    validar_politica()
except ValueError as e:
    print("Ruptura 1 (umbral):", str(e).split("- ")[-1])
finally:
    UMBRAL_COSTO_MINIMO = _u

# 2) Paro por límite: un tope demasiado bajo corta la corrida con constancia
tr_rup = Traza("C2-ruptura", tope_pasos=3)
for i in range(6):
    tr_rup.paso("observar", f"paso {i+1}")
tr_rup.mostrar()

# 3) Caso sin evidencia fechada: bloqueado antes de la bandeja
if top is None:
    print("Ruptura 3: NO EVALUADA; no hay candidato. No se fabrica uno.")
else:
    caso_sin_fecha = dict(tienda_id=top.tienda_id, producto_id=top.producto_id, costo=float(top.costo_estimado),
                          accion_propuesta="alerta a preventista", conflicto=False)
    print("\nRuptura 3 (sin fecha): criterios pasan:", verificar_criterios(caso_sin_fecha))

# 4) Una cadena no vacía no basta: la fecha debe existir y usar YYYY-MM-DD.
if top is not None:
    caso_fecha_invalida = dict(caso_sin_fecha, fecha_corte="NO ES UNA FECHA")
    print("Ruptura 4 (fecha inválida): criterios pasan:", verificar_criterios(caso_fecha_invalida))


Ruptura 1 (umbral): UMBRAL_COSTO_MINIMO debe ser un número finito positivo
----- Traza C2-ruptura -----
[ 1] OBSERVAR     paso 1
[ 2] OBSERVAR     paso 2
[ 3] PARO         PARO POR LIMITE: no queda espacio para otro paso y su cierre; tope 3 | evidencia: {'ultimo_registro_reservado': True}

Ruptura 3 (sin fecha): criterios pasan: False
Ruptura 4 (fecha inválida): criterios pasan: False


In [7]:
def pruebas_del_notebook():
    detalles = lambda tr: " | ".join(l["detalle"] for l in tr.lineas)
    if top is None:
        segura = q.empty and r1 == "imposibilidad" and not b1.casos and not b2.casos and not b3.casos
        print("Imposibilidad sin escrituras:", segura)
        print("NO EVALUADOS: éxito, escalamiento con candidato y ataque fuera de contrato.")
        print("Documenten el motivo; no fabriquen casos ni modifiquen umbrales solo para obtener verde.")
        return False
    resultados = [("parámetros básicos válidos y umbral aplicado al conjunto común",
                   POLITICA_OK and float(top.costo_estimado) >= UMBRAL_COSTO_MINIMO)]
    limite_seguro = tr1.paro_por_limite and r1 == "rechazada_limite" and not b1.casos
    bloqueo_seguro = r1 == "bloqueada_por_criterios" and not criterios_ok and not b1.casos
    escalamiento_seguro = r1 == "escalada" and conflicto and not b1.casos
    exito = r1 == "aceptada" and len(b1.casos) == 1 and "ÉXITO" in detalles(tr1)
    if limite_seguro:
        etiqueta = "ejecución 1: paro por límite sin escrituras; éxito NO EVALUADO"
    elif bloqueo_seguro:
        etiqueta = "ejecución 1: acción bloqueada por criterios; éxito NO EVALUADO"
    elif escalamiento_seguro:
        etiqueta = "ejecución 1: conflicto escalado sin escrituras; éxito NO EVALUADO"
    else:
        etiqueta = "ejecución 1: vigencias declaradas, criterios comprobados y paro por éxito"
    resultados.append((etiqueta, all(f in detalles(tr1) for f in VENTANAS_VIGENCIA)
                       and (limite_seguro or bloqueo_seguro or escalamiento_seguro or exito)))
    resultados.append(("ejecución 2: conflicto escalado e imposibilidad declarada",
                       r2 == "escalada" and not b2.casos and "IMPOSIBILIDAD" in detalles(tr2)))
    resultados.append(("adversarial: acción fuera de contrato bloqueada antes de escribir",
                       bloqueado is True and not b3.casos))
    aprobadas = sum(bool(ok) for _, ok in resultados)
    for nombre, ok in resultados:
        print(("[ok] " if ok else "[REVISAR] ") + nombre)
    print(f"\n{aprobadas}/4 comprobaciones del comportamiento observado satisfechas")
    print("Éxito de Ejecución 1:", "DEMOSTRADO" if exito else "NO DEMOSTRADO; documenten el estado y su causa")
    if aprobadas < 4:
        print("Revisen la comprobación señalada y su registro; distingan una falla de un caso no evaluado.")
    return aprobadas == 4

pruebas_del_notebook()


[ok] parámetros básicos válidos y umbral aplicado al conjunto común
[ok] ejecución 1: vigencias declaradas, criterios comprobados y paro por éxito
[ok] ejecución 2: conflicto escalado e imposibilidad declarada
[ok] adversarial: acción fuera de contrato bloqueada antes de escribir

4/4 comprobaciones del comportamiento observado satisfechas
Éxito de Ejecución 1: DEMOSTRADO


True

## Conclusiones

| Criterio de la rúbrica | Dónde quedó demostrado |
|---|---|
| Especificación verificable (30) | Decisiones + Ejecución 1: umbrales y criterios ejecutados; vigencia contabilizada, no aplicada como filtro |
| Coherencia con el contrato (25) | Adversarial: la acción prohibida no llega a la bandeja |
| Decisiones justificadas (20) | Su documento; aquí se ve «dónde verificar» operando |
| Evidencia con traza (15) | Ejecución 1 y Ejecución 2: resultado real, escalamiento e imposibilidad; control adversarial separado |
| Incertidumbre y escalamiento (10) | Capturas caducas declaradas; conflicto que escala con paquete |

Qué sigue: completen la autoevaluación de la plantilla señalando estas celdas como evidencia, e incorporen sus decisiones, resultados y trazas legibles en el PDF.

**Antes de cerrar el PDF:** expliquen qué decisión del contrato se volvió comprobable, qué muestra el resultado real y qué sigue siendo parte del diseño. El resultado de las pruebas no es una calificación ni prueba que todo el contrato esté implementado. Un paro justificado también aporta evidencia de aprendizaje.


In [8]:
# Declaración de uso de IA del equipo (igual que en el PDF)
# Si no usaron IA, declárenlo. Si no detectaron errores, indiquen qué comprobaron.
DECLARACION_IA = """
Qué le delegamos y con qué herramienta (o no utilizamos IA):
Qué verificamos:
Qué corregimos (o no identificamos errores tras las verificaciones descritas):
"""
